### Honey-bee colony loss: Modeling with all the features - weather + pathogen data

In [2]:
# Import necessary libraries

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, roc_auc_score, make_scorer, classification_report, r2_score, mean_absolute_error, mean_squared_error
from sklearn.utils import shuffle
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression,Ridge


#### Loading the dataset

Let us load the csv file containing our combined dataset. The file still has some features that we do not want to retain in the modeling part. We will proceed by dropping them. Additionally, let us also separate the master test set from the training set to avoid data leakage and final testing purposes. Our test set will consist of 2025 data and the last two quarters of 2024. 

In [3]:
# Loading the dataset
path = "../data/processed/bee_combined_all.csv"
final_df = pd.read_csv(path) 

# Dropping unnecessary columns
columns_to_remove = ['pct_pests_nomites','pct_disease', 'pct_other','pct_pesticides', 
                     'pct_unknown','unique_counties_sampled_q',
                     'unique_counties_sampled_q','last_month_in_quarter']
final_df = final_df.drop(columns=columns_to_remove)
final_df = final_df.sort_values(by=['state_code','year','quarter'], ascending=True).reset_index(drop=True)
TARGET_COLUMN = 'pct_loss_above_next_median'

# Separating the test set (2025 and last two quarters of 2024) from the training set
# Caution: 2025 data doesn't have target variable values for Q2 (makes sense since target is for next quarter)
condition = (final_df['year'] == 2025) | ((final_df['year'] == 2024) & ((final_df['quarter'] == 'Q3') | (final_df['quarter'] == 'Q4')))
test_df = final_df[condition].copy()

test_df.reset_index(drop=True, inplace=True)
test_df.to_csv("../data/processed/bee_combined_test_2025.csv", index=False)

train_df = final_df[~condition].copy()
train_df.reset_index(drop=True, inplace=True)
train_df.to_csv("../data/processed/bee_combined_train_until_2024Q2.csv", index=False)
final_df.head()

,Unnamed: 0,year,quarter,pct_loss,pct_varroa,state_code,region,mean_varroa_q,mean_spores_q,abpv_prevalence_q,...,Num_Frost_Days,M1_avg_tvol,M2_avg_tvol,M3_avg_tvol,M1_max_tvol,M2_max_tvol,M3_max_tvol,Q_tvol_avg,Num_high_vol_days,pct_loss_above_next_median
0,0,2015,Q1,26.0,10.0,AL,Southeast,NaN,NaN,NaN,...,0.430108,0.481383,0.625096,0.633761,0.256082,0.244946,0.340772,0.432035,0.085714,1.0
1,45,2015,Q2,12.0,16.7,AL,Southeast,2.366,0.143333,0.000,...,0.021505,0.439274,0.655469,0.615804,0.247119,0.425089,0.597105,0.422871,0.000000,1.0
2,90,2015,Q3,16.0,63.1,AL,Southeast,4.143,0.090000,0.125,...,0.010753,0.428967,0.598179,0.622202,0.451344,0.426873,0.276840,0.389428,0.000000,1.0
3,135,2015,Q4,8.0,3.1,AL,Southeast,9.085,0.000000,0.000,...,0.053763,0.518533,0.558070,0.572078,0.245839,0.233650,0.221954,0.379246,0.057143,1.0
4,180,2016,Q1,23.0,24.2,AL,Southeast,2.002,0.030000,0.000,...,0.301075,0.435561,0.645503,0.678748,0.210627,0.319263,0.438480,0.447457,0.028571,1.0


Let us make some quick checks on the NaN distribution and the target variable distribution in training and testing data

In [4]:
train_df.head()

,Unnamed: 0,year,quarter,pct_loss,pct_varroa,state_code,region,mean_varroa_q,mean_spores_q,abpv_prevalence_q,...,Num_Frost_Days,M1_avg_tvol,M2_avg_tvol,M3_avg_tvol,M1_max_tvol,M2_max_tvol,M3_max_tvol,Q_tvol_avg,Num_high_vol_days,pct_loss_above_next_median
0,0,2015,Q1,26.0,10.0,AL,Southeast,NaN,NaN,NaN,...,0.430108,0.481383,0.625096,0.633761,0.256082,0.244946,0.340772,0.432035,0.085714,1.0
1,45,2015,Q2,12.0,16.7,AL,Southeast,2.366,0.143333,0.000,...,0.021505,0.439274,0.655469,0.615804,0.247119,0.425089,0.597105,0.422871,0.000000,1.0
2,90,2015,Q3,16.0,63.1,AL,Southeast,4.143,0.090000,0.125,...,0.010753,0.428967,0.598179,0.622202,0.451344,0.426873,0.276840,0.389428,0.000000,1.0
3,135,2015,Q4,8.0,3.1,AL,Southeast,9.085,0.000000,0.000,...,0.053763,0.518533,0.558070,0.572078,0.245839,0.233650,0.221954,0.379246,0.057143,1.0
4,180,2016,Q1,23.0,24.2,AL,Southeast,2.002,0.030000,0.000,...,0.301075,0.435561,0.645503,0.678748,0.210627,0.319263,0.438480,0.447457,0.028571,1.0


In [5]:
# Removing rows with NaN target variable in training set
train_df = train_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
# only_pathogen_df = train_df.dropna(subset=['mean_varroa_q']).reset_index(drop=True)

# Defining feature sets
WEATHER_FEATURES = [
    'M1_avg_prcp', 'M2_avg_prcp', 'M3_avg_prcp', 'M1_max_prcp', 'M2_max_prcp',
    'M3_max_prcp', 'Q_prcp_avg', 'Consecutive_Rain_Days', 'M1_avg_tmax',
    'M2_avg_tmax', 'M3_avg_tmax', 'M1_max_tmax', 'M2_max_tmax', 'M3_max_tmax',
    'Q_tmax_avg', 'Num_Heat_Stress_Days', 'M1_avg_tmin', 'M2_avg_tmin',
    'M3_avg_tmin', 'M1_min_tmin', 'M2_min_tmin', 'M3_min_tmin', 'Q_tmin_avg',
    'Num_Frost_Days', 'M1_avg_tvol', 'M2_avg_tvol', 'M3_avg_tvol',
    'M1_max_tvol', 'M2_max_tvol', 'M3_max_tvol', 'Q_tvol_avg',
    'Num_high_vol_days'
]

PATHOGEN_FEATURES = [
    'mean_varroa_q', 'mean_spores_q', 'abpv_prevalence_q', 'cbpv_prevalence_q', 
    'dwv_prevalence_q', 'iapv_prevalence_q', 'kbv_prevalence_q', 'lsv2_prevalence_q', 
    'varroa_total_samples_q', 'spores_total_samples_q', 'abpv_total_samples_q', 
    'cbpv_total_samples_q', 'dwv_total_samples_q', 'iapv_total_samples_q', 
    'kbv_total_samples_q', 'lsv2_total_samples_q', 'worst_month_mean_varroa', 
    'worst_month_mean_spores', 'worst_month_abpv_prevalence', 'worst_month_cbpv_prevalence', 
    'worst_month_dwv_prevalence', 'worst_month_iapv_prevalence', 'worst_month_kbv_prevalence', 
    'worst_month_lsv2_prevalence', 'last_month_mean_varroa', 'last_month_mean_spores', 
    'last_month_abpv_prevalence', 'last_month_cbpv_prevalence', 'last_month_dwv_prevalence', 
    'last_month_iapv_prevalence', 'last_month_kbv_prevalence', 'last_month_lsv2_prevalence'
]

# Categorical features in our dataset
CATEGORICAL_FEATURES = ['year','quarter','state_code','region']
BASELINE_FEATURES = CATEGORICAL_FEATURES + WEATHER_FEATURES
ALL_NUMERICAL_FEATURES = WEATHER_FEATURES + PATHOGEN_FEATURES
ALL_FEATURES = BASELINE_FEATURES + PATHOGEN_FEATURES

# Preparing training and testing data for our model
X_train = train_df[ALL_FEATURES]
Y_train = train_df[TARGET_COLUMN]

X_test_final = test_df[ALL_FEATURES]
Y_test_final = test_df[TARGET_COLUMN]

In [6]:
from sklearn.impute import SimpleImputer

NUMERICAL_FEATURES = WEATHER_FEATURES + PATHOGEN_FEATURES

# Get the original training dataframe (assuming it's 'train_df')
df_train_to_check = train_df.copy()

# Impute ONLY the numerical columns
imputer = SimpleImputer(strategy='constant', fill_value=0)

# Fit and transform the numerical features
imputed_numerical_data = imputer.fit_transform(
    df_train_to_check[NUMERICAL_FEATURES]
)

# Create a new, purely numerical DataFrame 
df_imputed_numeric = pd.DataFrame(
    imputed_numerical_data, 
    columns=NUMERICAL_FEATURES,
    index=df_train_to_check.index
)

# Ensure the target is also numeric
df_imputed_numeric[TARGET_COLUMN] = pd.to_numeric(df_train_to_check[TARGET_COLUMN])

print("Calculating correlation matrix...")
corr_matrix = df_imputed_numeric.corr()

# Isolate just the target variable and sort
target_correlations = corr_matrix[TARGET_COLUMN].sort_values(ascending=False)

print("\n--- Correlation with Target Variable ---")
# Print the target's correlation with itself (should be 1.0) and the top 10 features
print(target_correlations.head(20))
print(target_correlations.tail(20)) 

Calculating correlation matrix...

--- Correlation with Target Variable ---
pct_loss_above_next_median    1.000000
M3_avg_tmax                   0.109985
M3_max_tvol                   0.100686
M3_max_tmax                   0.100216
M3_avg_tmin                   0.097216
M3_min_tmin                   0.093278
M3_avg_tvol                   0.087324
Q_tvol_avg                    0.065006
Num_high_vol_days             0.061262
Q_tmax_avg                    0.059644
M2_max_tmax                   0.057485
M2_max_tvol                   0.055191
M2_avg_tmax                   0.054232
M2_avg_tvol                   0.048848
Q_tmin_avg                    0.048127
Q_prcp_avg                    0.046446
M3_avg_prcp                   0.045788
M2_avg_tmin                   0.045737
M2_avg_prcp                   0.044045
M2_min_tmin                   0.039374
Name: pct_loss_above_next_median, dtype: float64
worst_month_iapv_prevalence   -0.023514
spores_total_samples_q        -0.024347
Num_Heat_Stress

In [7]:
# REmoving columns with less correlations than 0.1 and -0.1
low_correlation_features = target_correlations[(abs(target_correlations) < 0.1) & (target_correlations.index != TARGET_COLUMN)].index.tolist()
print("\nFeatures to be removed due to low correlation with target:")
print(low_correlation_features)
# X_train = X_train.drop(columns=low_correlation_features)
set1 = set(NUMERICAL_FEATURES)
set2 = set(low_correlation_features)

result_set = set1 - set2
REMAINING_FEATURES = list(result_set)
Y_train.value_counts()


Features to be removed due to low correlation with target:
['M3_avg_tmin', 'M3_min_tmin', 'M3_avg_tvol', 'Q_tvol_avg', 'Num_high_vol_days', 'Q_tmax_avg', 'M2_max_tmax', 'M2_max_tvol', 'M2_avg_tmax', 'M2_avg_tvol', 'Q_tmin_avg', 'Q_prcp_avg', 'M3_avg_prcp', 'M2_avg_tmin', 'M2_avg_prcp', 'M2_min_tmin', 'M1_avg_tvol', 'Consecutive_Rain_Days', 'M1_max_prcp', 'M1_avg_prcp', 'worst_month_kbv_prevalence', 'last_month_kbv_prevalence', 'M1_max_tvol', 'kbv_prevalence_q', 'M2_max_prcp', 'lsv2_prevalence_q', 'M1_avg_tmax', 'M3_max_prcp', 'cbpv_prevalence_q', 'M1_min_tmin', 'M1_avg_tmin', 'worst_month_lsv2_prevalence', 'last_month_lsv2_prevalence', 'mean_spores_q', 'M1_max_tmax', 'last_month_iapv_prevalence', 'worst_month_cbpv_prevalence', 'iapv_prevalence_q', 'worst_month_mean_spores', 'last_month_mean_spores', 'last_month_cbpv_prevalence', 'worst_month_iapv_prevalence', 'spores_total_samples_q', 'Num_Heat_Stress_Days', 'varroa_total_samples_q', 'last_month_dwv_prevalence', 'dwv_total_samples_q',

pct_loss_above_next_median
0.0    987
1.0    704
Name: count, dtype: int64

#### Creating pipeline to avoid data leakage

Now that we have separated our training and testing set, let us proceed with encoding our categorical features. One way to encode them for our model is through one-hot encoding scheme which would create new features based on the exisiting categorical feature. For example, the quarter feature would be encoded into four new columns (Q1, Q2, Q3, Q4). A row that was originally 'Q1' would be represented as [1, 0, 0, 0] across these new columns, while a 'Q2' row would be [0, 1, 0, 0].

Caution: One has to be very careful not to create data leakage when using the one-hot encoding scheme. For instance, if we apply this scheme on our entire training set and then proceed to create validation time series split, there could be potential data leakage from the later years to the initial folds consisting of initial years. To elaborate with an illustrative example, suppose our first fold consists of training set from 2015-16 and validation set from 2017. Consider a case where the Hawaii data exists only from 2017. If we directly use one-hot encoding scheme, there would be a column in the training set with state_Hawaii set to [0,...0]. Our model built using 2015-2016 data is already trained expecting Hawaii state feature which we want to avoid. 

Pipeline from scikit-learn is a nice module that strictly enforces splitting before the model training and makes our lives easier. We will be using this feature below.

In [9]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', MinMaxScaler())
])

# We use OneHotEncoder and tell it to ignore unknown categories
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Create a preprocessor that applies transformers to the correct columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
        # We tell it to leave all other features (our weather data) alone
        ('num', numerical_transformer, REMAINING_FEATURES)
    ],
    remainder='passthrough' # Keep any other columns
)

model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

# Create the full pipeline - First preprocesses the data, then feeds it to the model
# full_model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),('model', model)])
regression_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Defining cross-validation strategy
tscv = TimeSeriesSplit(n_splits=5)

Now that we have defined our pipeline for handling preprocessing and initiated our lightgbm model for classification, let us proceed with defining KPIs for evaluating our model's performance. Fianlly, let us also run the cross-validation.

In [10]:
# Define the scorers for our KPIs
scorers = {'f1_score': make_scorer(f1_score, pos_label=1),'roc_auc': make_scorer(roc_auc_score)}

print("Running 5-fold Time-Series Cross-Validation for Regression...")

cv_results = cross_validate(
    regression_pipeline,
    X_train,
    Y_train,
    cv=tscv, # Your TimeSeriesSplit
    scoring=scorers,
    n_jobs=-1
)

print("\n--- Cross-Validation Results ---")
print(f"Average F1 Score:   {np.mean(cv_results['test_f1_score']):.3f} +/- {np.std(cv_results['test_f1_score']):.3f}")
print(f"Average ROC-AUC Score: {np.mean(cv_results['test_roc_auc']):.3f} +/- {np.std(cv_results['test_roc_auc']):.3f}")
print("\nIndividual Fold Scores:")
print(f"F1:   {cv_results['test_f1_score']}")
print(f"ROC-AUC: {cv_results['test_roc_auc']}")


Running 5-fold Time-Series Cross-Validation for Regression...
[LightGBM] [Info] Number of positive: 608, number of negative: 802
[LightGBM] [Info] Number of positive: 267, number of negative: 300[LightGBM] [Info] Number of positive: 504, number of negative: 625[LightGBM] [Info] Number of positive: 388, number of negative: 460


[LightGBM] [Info] Number of positive: 137, number of negative: 149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001148 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3590
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001196 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is no

/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Cross-Validation Results ---
Average F1 Score:   0.470 +/- 0.034
Average ROC-AUC Score: 0.562 +/- 0.029

Individual Fold Scores:
F1:   [0.52509653 0.41921397 0.47787611 0.46601942 0.46236559]
ROC-AUC: [0.55955171 0.51084711 0.56306165 0.57822686 0.59693131]


/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/d2l/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
print(X_train[PATHOGEN_FEATURES].describe())

       mean_varroa_q  mean_spores_q  abpv_prevalence_q  cbpv_prevalence_q  \
count     843.000000     843.000000         847.000000         847.000000   
mean        2.828238       0.266580           0.172391           0.122580   
std         2.954252       0.384531           0.257100           0.178935   
min         0.000000       0.000000           0.000000           0.000000   
25%         0.851000       0.036039           0.000000           0.000000   
50%         2.064000       0.137500           0.058824           0.066667   
75%         3.655000       0.341667           0.250000           0.187500   
max        23.430000       3.175000           1.000000           1.000000   

       dwv_prevalence_q  iapv_prevalence_q  kbv_prevalence_q  \
count        847.000000         847.000000        847.000000   
mean           0.799028           0.217684          0.026979   
std            0.232448           0.226396          0.093315   
min            0.000000           0.000000        

**Takeaways:** Our model scores are pretty low which indicated the model might by underfitting. The model is not really able to capture any signals in our dataset. The ultimate question remains: Is there any signal in our dataset or the coarse graining at the state level and quarterly level is washing out meaningful signals? 